# Master Training Pipeline: Optuna Hybrid
**Architecture:** Tri-Layer Hybrid (Overfitting-Free)
1. KNN Imputation (Zero Leakage Missing Data Handling)
2. Isolation Forest + IQR (Anomaly Detection & Imputation)
3. Prophet (Base Macroscopic Trending + Weather Regressor) — **Train-Only**
4. LightGBM (Micro Residual Corrections, Regularised) — **Train-Only, Val = Held-Out**

**Key Design Decisions:**
- Both Prophet and LightGBM train strictly on `train_df` to prevent data leakage.
- Validation set is used *only* for early-stopping and evaluation — never for training.
- LightGBM uses `extra_trees`, L1/L2 regularisation, and restricted tree depth to prevent overfitting.
- Diagnostic output separates **Overfitting** (Train→Val gap) from **Distribution Shift** (Val→Test gap).

In [ ]:
import os
import json
import pandas as pd
import numpy as np
import matplotlib
import optuna
import logging
import matplotlib.pyplot as plt
from prophet import Prophet
from sklearn.ensemble import IsolationForest
from sklearn.metrics import mean_squared_error, mean_absolute_error
from sklearn.metrics import precision_score, recall_score, f1_score
from sklearn.impute import KNNImputer
import lightgbm as lgb
import shap
import joblib
import warnings

warnings.filterwarnings('ignore')

# Resolve paths relative to this notebook
PROJECT_ROOT = os.path.abspath('..')

output_dir = os.path.join(PROJECT_ROOT, 'Outputs')
models_dir = os.path.join(PROJECT_ROOT, 'Models')
os.makedirs(models_dir, exist_ok=True)
os.makedirs(output_dir, exist_ok=True)

params_path = os.path.join(models_dir, 'best_hybrid_params.json')

# Tuning control
OPTUNA_TRIALS = int(os.getenv('OPTUNA_TRIALS', '50'))
RETUNE_EVERY_DAYS = int(os.getenv('RETUNE_EVERY_DAYS', '30'))
FORCE_RETUNE = True

## 1. Pre-processing & Data Ingestion

In [ ]:
print("1. Pre-processing: Raw data ingestion...")
train_dir = os.path.join(PROJECT_ROOT, 'train_data')
test_dir = os.path.join(PROJECT_ROOT, 'test_data')

train_df = pd.read_csv(os.path.join(train_dir, 'dataset_daily_train.csv'))
val_df = pd.read_csv(os.path.join(test_dir, 'dataset_daily_val.csv'))
test_df = pd.read_csv(os.path.join(test_dir, 'dataset_daily_test.csv'))

train_df['Date'] = pd.to_datetime(train_df['Date'])
val_df['Date'] = pd.to_datetime(val_df['Date'])
test_df['Date'] = pd.to_datetime(test_df['Date'])

print(f"   Train: {train_df.shape}, Val: {val_df.shape}, Test: {test_df.shape}")

## 2. Zero-Leakage ML Imputation (KNNImputer)

In [ ]:
print("2. Handling missing values via KNN Imputation (Fit on Train, Apply to all)...")
for df_part in [train_df, val_df, test_df]:
    df_part['Time_Idx'] = df_part['Date'].dt.dayofyear

features_to_impute = ['Time_Idx', 'Demand_MWh', 'Avg_Temp', 'Rainfall']

imputer = KNNImputer(n_neighbors=5, weights='distance')
# 1. FIT STRICTLY ON TRAIN
imputer.fit(train_df[features_to_impute])

# 2. APPLY TO ALL PARTITIONS 
train_df[features_to_impute] = imputer.transform(train_df[features_to_impute])
val_df[features_to_impute]   = imputer.transform(val_df[features_to_impute])
test_df[features_to_impute]  = imputer.transform(test_df[features_to_impute])

for df_part in [train_df, val_df, test_df]:
    df_part.drop(columns=['Time_Idx'], inplace=True)

joblib.dump(imputer, os.path.join(models_dir, 'knn_imputer.joblib'))
print("   KNN Imputer saved.")

## 3. Feature Engineering (18 Extended Features)

In [ ]:
print("3. Feature engineering & cleanup...")
target_col = 'Demand_MWh'

for df_part in [train_df, val_df, test_df]:
    df_part['Month']      = df_part['Date'].dt.month
    df_part['DayOfYear']  = df_part['Date'].dt.dayofyear
    df_part['WeekOfYear'] = df_part['Date'].dt.isocalendar().week.astype(int)
    df_part['Trend'] = (df_part['Date'] - pd.Timestamp('2018-01-01')).dt.days
    df_part['Lag_2']  = df_part[target_col].shift(2)
    df_part['Lag_14'] = df_part[target_col].shift(14)
    df_part['Rolling_14'] = df_part[target_col].rolling(window=14, min_periods=1).mean()
    df_part['Rolling_30'] = df_part[target_col].rolling(window=30, min_periods=1).mean()
    df_part['Temp_Lag_1'] = df_part['Avg_Temp'].shift(1)

features = [col for col in [
    'Day_of_Week', 'Is_Weekend', 'Is_Holiday',
    'Month', 'DayOfYear', 'WeekOfYear', 'Trend',
    'Avg_Temp', 'Rainfall', 'Temp_Lag_1',
    'Lag_1', 'Lag_2', 'Lag_7', 'Lag_14', 'Lag_30',
    'Rolling_7', 'Rolling_14', 'Rolling_30',
] if col in train_df.columns]

train_df = train_df.dropna(subset=features + [target_col]).copy()
val_df = val_df.dropna(subset=features + [target_col]).copy()
test_df = test_df.dropna(subset=features + [target_col]).copy()

df = pd.concat([train_df, val_df, test_df]).sort_values('Date').reset_index(drop=True)

print(f"   Available features ({len(features)}): {features}")
print(f"   Train: {len(train_df)} | Val: {len(val_df)} | Test: {len(test_df)}")

## 4. Joint Bayesian Optimization (Optuna)
Searches for the best hyperparameters across the entire pipeline:
- Isolation Forest contamination
- Prophet changepoint/seasonality priors
- LightGBM tree structure + regularisation (L1, L2, min_child_samples)

In [ ]:
print("4. Joint Bayesian Optimization (Optuna)...")

optuna.logging.set_verbosity(optuna.logging.WARNING)
logger = logging.getLogger('cmdstanpy')
logger.addHandler(logging.NullHandler())
logger.propagate = False
logger.setLevel(logging.CRITICAL)

required_param_keys = {
    'contamination', 'changepoint_prior_scale', 'seasonality_prior_scale',
    'n_changepoints', 'learning_rate', 'max_depth', 'num_leaves',
    'subsample', 'colsample_bytree', 'min_child_samples', 'reg_alpha', 'reg_lambda'
}

def load_saved_params(path):
    if not os.path.exists(path):
        return None
    try:
        with open(path, 'r', encoding='utf-8') as f:
            payload = json.load(f)
        params = payload.get('best_params', payload)
        if not isinstance(params, dict) or not required_param_keys.issubset(set(params.keys())):
            return None
        return payload
    except Exception as e:
        print(f"   Warning: failed to read saved params ({e}).")
        return None

def should_retune(saved_payload):
    if FORCE_RETUNE:
        print("   Retune reason: FORCE_RETUNE=True")
        return True
    if saved_payload is None:
        print("   Retune reason: no saved parameter file found.")
        return True
    tuned_at = saved_payload.get('last_tuned_at')
    if not tuned_at:
        return True
    try:
        age_days = (pd.Timestamp.now() - pd.to_datetime(tuned_at)).days
        if age_days >= RETUNE_EVERY_DAYS:
            print(f"   Retune reason: params age {age_days} days >= {RETUNE_EVERY_DAYS} days.")
            return True
        print(f"   Using saved params (age: {age_days} days).")
        return False
    except Exception:
        return True

df_prophet_val_proxy = val_df[['Date', 'Avg_Temp']].rename(columns={'Date': 'ds'})

def objective(trial):
    contamination = trial.suggest_float('contamination', 0.001, 0.05, log=True)
    cps = trial.suggest_float('changepoint_prior_scale', 0.001, 0.1, log=True)
    sps = trial.suggest_float('seasonality_prior_scale', 0.01, 1.0, log=True)
    n_cp = trial.suggest_int('n_changepoints', 5, 20)
    
    lgb_lr = trial.suggest_float('learning_rate', 0.001, 0.05, log=True)
    lgb_depth = trial.suggest_int('max_depth', 2, 4)
    lgb_leaves = trial.suggest_int('num_leaves', 4, 15)
    lgb_subsample = trial.suggest_float('subsample', 0.4, 0.8)
    lgb_colsample = trial.suggest_float('colsample_bytree', 0.4, 0.8)
    lgb_min_child = trial.suggest_int('min_child_samples', 15, 60)
    lgb_reg_alpha = trial.suggest_float('reg_alpha', 0.01, 10.0, log=True)
    lgb_reg_lambda = trial.suggest_float('reg_lambda', 0.01, 10.0, log=True)
    
    # Anomaly Detection + Imputation
    temp_forest = IsolationForest(n_estimators=100, max_samples='auto', contamination=contamination, random_state=42, n_jobs=-1)
    temp_forest.fit(train_df[features])
    temp_anomalies = temp_forest.predict(train_df[features])
    Q1_tmp = train_df[target_col].quantile(0.25)
    Q3_tmp = train_df[target_col].quantile(0.75)
    IQR_tmp = Q3_tmp - Q1_tmp
    iqr_anomalies_tmp = np.where((train_df[target_col] < Q1_tmp - 1.5*IQR_tmp) | (train_df[target_col] > Q3_tmp + 1.5*IQR_tmp), -1, 1)
    is_anomaly_tmp = (temp_anomalies == -1) | (iqr_anomalies_tmp == -1)
    temp_train_clean = train_df.copy()
    for idx in np.where(is_anomaly_tmp)[0]:
        start = max(0, idx - 7)
        clean_window = train_df[target_col].iloc[start:idx][~is_anomaly_tmp[start:idx]]
        temp_train_clean.iloc[idx, temp_train_clean.columns.get_loc(target_col)] = clean_window.mean() if len(clean_window) > 0 else train_df[target_col].mean()
    
    # Prophet (train-only)
    df_prophet_temp = temp_train_clean[['Date', target_col, 'Avg_Temp']].rename(columns={'Date': 'ds', target_col: 'y'})
    m_base = Prophet(yearly_seasonality=True, weekly_seasonality=True, daily_seasonality=False, changepoint_prior_scale=cps, seasonality_prior_scale=sps, n_changepoints=n_cp)
    m_base.add_regressor('Avg_Temp')
    m_base.fit(df_prophet_temp)
    temp_train_clean['Prophet_Pred'] = m_base.predict(df_prophet_temp)['yhat'].values
    temp_val = val_df.copy()
    temp_val['Prophet_Pred'] = m_base.predict(df_prophet_val_proxy)['yhat'].values
    
    # LightGBM (train-only, val for early stopping)
    temp_train_clean['Prophet_Residual'] = temp_train_clean[target_col] - temp_train_clean['Prophet_Pred']
    temp_val['Prophet_Residual'] = temp_val[target_col] - temp_val['Prophet_Pred']
    lgb_model = lgb.LGBMRegressor(
        learning_rate=lgb_lr, max_depth=lgb_depth, num_leaves=lgb_leaves,
        subsample=lgb_subsample, colsample_bytree=lgb_colsample,
        min_child_samples=lgb_min_child, reg_alpha=lgb_reg_alpha, reg_lambda=lgb_reg_lambda,
        n_estimators=800, random_state=42, n_jobs=-1, verbose=-1, extra_trees=True
    )
    lgb_model.fit(
        temp_train_clean[features], temp_train_clean['Prophet_Residual'],
        eval_set=[(temp_val[features], temp_val['Prophet_Residual'])],
        callbacks=[lgb.early_stopping(stopping_rounds=20, verbose=False), lgb.log_evaluation(period=0)]
    )
    hybrid_preds = temp_val['Prophet_Pred'] + lgb_model.predict(temp_val[features])
    return mean_absolute_error(temp_val[target_col], hybrid_preds)

saved_payload = load_saved_params(params_path)
if should_retune(saved_payload):
    print(f"   Running Joint Bayesian Search ({OPTUNA_TRIALS} Trials)...")
    sampler = optuna.samplers.TPESampler(seed=0)
    study = optuna.create_study(direction='minimize', sampler=sampler)
    study.optimize(objective, n_trials=OPTUNA_TRIALS, show_progress_bar=True)
    best_p = study.best_params
    print(f"   Best Joint Parameters found: {best_p}")
    params_payload = {'best_params': best_p, 'last_tuned_at': pd.Timestamp.now().isoformat(),
                      'n_trials': OPTUNA_TRIALS, 'retune_every_days': RETUNE_EVERY_DAYS,
                      'target_col': target_col, 'features': features}
    with open(params_path, 'w', encoding='utf-8') as f:
        json.dump(params_payload, f, indent=2)
    print(f"   Saved best parameters to: {params_path}")
else:
    best_p = saved_payload['best_params']
    print(f"   Loaded saved best parameters from: {params_path}")

## 5. Training Final Champion Model
Both Prophet and LightGBM are trained **strictly on `train_df`** only.
- Val is used only as early-stopping monitor for LightGBM.
- Neither model has ever seen `val_df` or `test_df` during training.

In [ ]:
print("5. Training Final Champion Architecture...")

# --- 5a. Anomaly Detection + Imputation (IQR + Isolation Forest) ---
iso_forest = IsolationForest(n_estimators=300, max_samples='auto', contamination=best_p['contamination'], random_state=42, n_jobs=-1)
iso_forest.fit(train_df[features])
train_anomalies = iso_forest.predict(train_df[features])

Q1 = train_df[target_col].quantile(0.25)
Q3 = train_df[target_col].quantile(0.75)
IQR_val = Q3 - Q1
iqr_anomalies = np.where((train_df[target_col] < Q1 - 1.5*IQR_val) | (train_df[target_col] > Q3 + 1.5*IQR_val), -1, 1)
is_anomaly = (train_anomalies == -1) | (iqr_anomalies == -1)
num_anomalies = is_anomaly.sum()

train_df_clean = train_df.copy()
for idx in np.where(is_anomaly)[0]:
    start = max(0, idx - 7)
    clean_window = train_df[target_col].iloc[start:idx][~is_anomaly[start:idx]]
    train_df_clean.iloc[idx, train_df_clean.columns.get_loc(target_col)] = clean_window.mean() if len(clean_window) > 0 else train_df[target_col].mean()

print(f"   Imputed {num_anomalies} anomalies (0 rows removed).")

# IF evaluation
y_true_anom = (iqr_anomalies == -1).astype(int)
y_pred_anom = (train_anomalies == -1).astype(int)
print(f"   IF vs IQR — P: {precision_score(y_true_anom, y_pred_anom, zero_division=0):.3f}, R: {recall_score(y_true_anom, y_pred_anom, zero_division=0):.3f}, F1: {f1_score(y_true_anom, y_pred_anom, zero_division=0):.3f}")

# --- 5b. Final Prophet Training (train-only) ---
print("   Training Final Prophet (train-only)...")
df_prophet_train = train_df_clean[['Date', target_col, 'Avg_Temp']].rename(columns={'Date': 'ds', target_col: 'y'})
prophet_model = Prophet(
    yearly_seasonality=True, weekly_seasonality=True, daily_seasonality=False,
    changepoint_prior_scale=best_p['changepoint_prior_scale'],
    seasonality_prior_scale=best_p['seasonality_prior_scale'],
    n_changepoints=best_p['n_changepoints']
)
prophet_model.add_regressor('Avg_Temp')
prophet_model.fit(df_prophet_train)

for split_df in [train_df_clean, train_df, val_df, test_df]:
    future = split_df[['Date', 'Avg_Temp']].rename(columns={'Date': 'ds'})
    split_df['Prophet_Pred'] = prophet_model.predict(future)['yhat'].values

# --- 5c. Final LightGBM Training (train-only, val = held-out early-stop) ---
print("   Training Final LightGBM (train-only, val = held-out)...")
train_df_clean['Prophet_Residual'] = train_df_clean[target_col] - train_df_clean['Prophet_Pred']
train_df['Prophet_Residual'] = train_df[target_col] - train_df['Prophet_Pred']
val_df['Prophet_Residual'] = val_df[target_col] - val_df['Prophet_Pred']
test_df['Prophet_Residual'] = test_df[target_col] - test_df['Prophet_Pred']

model_lgb = lgb.LGBMRegressor(
    learning_rate=best_p['learning_rate'], max_depth=best_p['max_depth'],
    num_leaves=best_p['num_leaves'], subsample=best_p['subsample'],
    colsample_bytree=best_p['colsample_bytree'],
    min_child_samples=best_p['min_child_samples'],
    reg_alpha=best_p['reg_alpha'], reg_lambda=best_p['reg_lambda'],
    n_estimators=800, random_state=42, n_jobs=-1, verbose=-1, extra_trees=True
)
model_lgb.fit(
    train_df_clean[features], train_df_clean['Prophet_Residual'],
    eval_set=[(val_df[features], val_df['Prophet_Residual'])],
    callbacks=[lgb.early_stopping(stopping_rounds=50, verbose=False), lgb.log_evaluation(period=0)]
)

for split_df in [train_df, val_df, test_df]:
    split_df['LGBM_Residual_Pred'] = model_lgb.predict(split_df[features])

train_df['Final_Pred'] = train_df['Prophet_Pred'] + train_df['LGBM_Residual_Pred']
val_df['Final_Pred'] = val_df['Prophet_Pred'] + val_df['LGBM_Residual_Pred']
test_df['Final_Pred'] = test_df['Prophet_Pred'] + test_df['LGBM_Residual_Pred']
print("   Final Champion Architecture trained.")

## 6. Model Evaluation (Train / Val / Test)
- **Train**: In-sample for both Prophet and LightGBM (naturally low error)
- **Val**: Fully out-of-sample for both models (honest generalisation)
- **Test**: Fully out-of-sample for both models (real-world performance)

**Train→Val gap** = Overfitting indicator | **Val→Test gap** = Distribution Shift indicator

In [ ]:
print("6. Evaluating Model Performance...")

def calc_mape(actual, predicted):
    mask = actual != 0
    return np.mean(np.abs((actual[mask] - predicted[mask]) / actual[mask])) * 100

# Prophet-Only Metrics
prophet_mae_train  = mean_absolute_error(train_df['Demand_MWh'], train_df['Prophet_Pred'])
prophet_rmse_train = np.sqrt(mean_squared_error(train_df['Demand_MWh'], train_df['Prophet_Pred']))
prophet_mape_train = calc_mape(train_df['Demand_MWh'].values, train_df['Prophet_Pred'].values)
prophet_mae_val  = mean_absolute_error(val_df['Demand_MWh'], val_df['Prophet_Pred'])
prophet_rmse_val = np.sqrt(mean_squared_error(val_df['Demand_MWh'], val_df['Prophet_Pred']))
prophet_mape_val = calc_mape(val_df['Demand_MWh'].values, val_df['Prophet_Pred'].values)
prophet_mae_test  = mean_absolute_error(test_df['Demand_MWh'], test_df['Prophet_Pred'])
prophet_rmse_test = np.sqrt(mean_squared_error(test_df['Demand_MWh'], test_df['Prophet_Pred']))
prophet_mape_test = calc_mape(test_df['Demand_MWh'].values, test_df['Prophet_Pred'].values)

# Hybrid Metrics
hybrid_mae_train  = mean_absolute_error(train_df['Demand_MWh'], train_df['Final_Pred'])
hybrid_rmse_train = np.sqrt(mean_squared_error(train_df['Demand_MWh'], train_df['Final_Pred']))
hybrid_mape_train = calc_mape(train_df['Demand_MWh'].values, train_df['Final_Pred'].values)
hybrid_mae_val  = mean_absolute_error(val_df['Demand_MWh'], val_df['Final_Pred'])
hybrid_rmse_val = np.sqrt(mean_squared_error(val_df['Demand_MWh'], val_df['Final_Pred']))
hybrid_mape_val = calc_mape(val_df['Demand_MWh'].values, val_df['Final_Pred'].values)
hybrid_mae_test  = mean_absolute_error(test_df['Demand_MWh'], test_df['Final_Pred'])
hybrid_rmse_test = np.sqrt(mean_squared_error(test_df['Demand_MWh'], test_df['Final_Pred']))
hybrid_mape_test = calc_mape(test_df['Demand_MWh'].values, test_df['Final_Pred'].values)

print("\n" + "=" * 82)
print("  MODEL COMPARISON: Prophet-Only vs Hybrid (Prophet + LightGBM)")
print("=" * 82)
for label, suffix, note in [('Train', 'train', 'In-sample'), ('Val', 'val', 'Out-of-sample'), ('Test', 'test', 'Out-of-sample')]:
    p_mae = eval(f'prophet_mae_{suffix}'); p_rmse = eval(f'prophet_rmse_{suffix}'); p_mape = eval(f'prophet_mape_{suffix}')
    h_mae = eval(f'hybrid_mae_{suffix}'); h_rmse = eval(f'hybrid_rmse_{suffix}'); h_mape = eval(f'hybrid_mape_{suffix}')
    print(f"{'Metric':<12} | {f'Prophet-Only ({label})':<22} | {f'Hybrid ({label})':<22} | {note}")
    print("-" * 82)
    print(f"{'MAE':<12} | {p_mae:>18,.2f} MWh | {h_mae:>18,.2f} MWh |")
    print(f"{'RMSE':<12} | {p_rmse:>18,.2f} MWh | {h_rmse:>18,.2f} MWh |")
    print(f"{'MAPE':<12} | {p_mape:>17.2f}%     | {h_mape:>17.2f}%     |")
    print("-" * 82)
print("=" * 82)

# Diagnostics
overfit_gap = hybrid_mape_val - hybrid_mape_train
shift_gap   = hybrid_mape_test - hybrid_mape_val
print(f"\n  >> Train→Val Gap:  {overfit_gap:.2f}pp  (Overfitting indicator)")
print(f"  >> Val→Test Gap:   {shift_gap:.2f}pp  (Distribution Shift indicator)")
if overfit_gap < 1.0:
    print("  >> OVERFIT CHECK:  ✅ Healthy")
elif overfit_gap < 2.5:
    print("  >> OVERFIT CHECK:  ⚠️  Mild overfitting")
else:
    print("  >> OVERFIT CHECK:  ❌ Significant overfitting")
if shift_gap < 0.5:
    print("  >> SHIFT CHECK:    ✅ Stable")
elif shift_gap < 2.0:
    print("  >> SHIFT CHECK:    ⚠️  Moderate distribution shift")
else:
    print("  >> SHIFT CHECK:    ❌ Large distribution shift")

rmse_improvement = ((prophet_rmse_test - hybrid_rmse_test) / prophet_rmse_test) * 100
mape_improvement = ((prophet_mape_test - hybrid_mape_test) / prophet_mape_test) * 100
print(f"\n  >> Hybrid improves Test RMSE by {rmse_improvement:.1f}%")
print(f"  >> Hybrid improves Test MAPE by {mape_improvement:.1f}%")

## 7. Serializing Models & Predictions

In [ ]:
print("7. Exporting trained models and predictions...")

joblib.dump(prophet_model, os.path.join(models_dir, 'prophet_model.joblib'))
joblib.dump(model_lgb, os.path.join(models_dir, 'lgbm_model.joblib'))
joblib.dump(iso_forest, os.path.join(models_dir, 'iso_forest.joblib'))
print(f"   Models saved to: {models_dir}")

df_all = pd.concat([train_df, val_df, test_df]).sort_values('Date').reset_index(drop=True)
df_all.rename(columns={'Final_Pred': 'Hybrid_Prediction'}, inplace=True)
predictions_path = os.path.join(output_dir, 'dataset_daily_with_predictions.csv')
df_all.to_csv(predictions_path, index=False)
print(f"   Predictions saved to: {predictions_path}")

## 8. Visualizations

In [ ]:
print("8. Generating Visualizations...")

plt.rcParams.update({
    'figure.facecolor': '#0f1117', 'axes.facecolor': '#1a1d29',
    'axes.edgecolor': '#2d3250', 'axes.labelcolor': '#e0e0e0',
    'text.color': '#e0e0e0', 'xtick.color': '#a0a0a0',
    'ytick.color': '#a0a0a0', 'grid.color': '#2d3250', 'grid.alpha': 0.5,
    'font.family': 'sans-serif', 'font.size': 10,
})

# Anomaly plot
anomalous_data = train_df[train_anomalies == -1]
plt.figure(figsize=(15, 6))
plt.plot(train_df['Date'], train_df[target_col], color='royalblue', label='Normal Demand', alpha=0.6, linewidth=1)
plt.scatter(anomalous_data['Date'], anomalous_data[target_col], color='crimson', label='Detected Anomaly', zorder=5)
plt.title('Isolation Forest: Detected Anomalies in Training Data')
plt.xlabel('Date'); plt.ylabel('Electricity Demand (MWh)'); plt.legend(); plt.tight_layout()
plt.savefig(os.path.join(output_dir, 'fig0_anomalies_detected.png'), dpi=300); plt.show()

# Full timeline
all_dates = pd.concat([train_df['Date'], val_df['Date'], test_df['Date']])
all_actual = pd.concat([train_df['Demand_MWh'], val_df['Demand_MWh'], test_df['Demand_MWh']])
all_prophet = pd.concat([train_df['Prophet_Pred'], val_df['Prophet_Pred'], test_df['Prophet_Pred']])
all_hybrid = pd.concat([train_df.get('Hybrid_Prediction', train_df['Final_Pred']),
                         val_df.get('Hybrid_Prediction', val_df['Final_Pred']),
                         test_df.get('Hybrid_Prediction', test_df['Final_Pred'])])
fig1, ax1 = plt.subplots(figsize=(16, 6))
ax1.plot(all_dates, all_actual, color='#4fc3f7', alpha=0.6, linewidth=0.7, label='Actual Demand')
ax1.plot(all_dates, all_prophet, color='#ff8a65', linewidth=0.9, linestyle='--', alpha=0.7, label='Prophet-Only')
ax1.plot(all_dates, all_hybrid, color='#66bb6a', linewidth=1.0, alpha=0.85, label='Hybrid (Prophet+LGB)')
ax1.axvspan(train_df['Date'].iloc[0], train_df['Date'].iloc[-1], alpha=0.04, color='#4fc3f7', label='Train (70%)')
ax1.axvspan(val_df['Date'].iloc[0], val_df['Date'].iloc[-1], alpha=0.08, color='#ffab40', label='Validation (15%)')
ax1.axvspan(test_df['Date'].iloc[0], test_df['Date'].iloc[-1], alpha=0.08, color='#ef5350', label='Test (15%)')
ax1.set_title('Electricity Demand: Actual vs Model Predictions (Full Timeline)', fontsize=14, fontweight='bold', pad=15)
ax1.set_xlabel('Date'); ax1.set_ylabel('Demand (MWh)')
ax1.legend(loc='upper left', fontsize=8, ncol=3, framealpha=0.3); ax1.grid(True, alpha=0.3)
fig1.tight_layout(); fig1.savefig(os.path.join(output_dir, 'fig1_actual_vs_predicted.png'), dpi=150, bbox_inches='tight')
plt.show()

## 9. Explainable AI (SHAP)

In [ ]:
print("9. Generating XAI SHAP Explanation...")
X_test = test_df[features]
explainer = shap.TreeExplainer(model_lgb)
shap_values = explainer.shap_values(X_test)
shap.summary_plot(shap_values, X_test, feature_names=features, show=True)

print("\n" + "=" * 72)
print("  SUCCESS! Training Pipeline Completed.")
print(f"    Models  -> {models_dir}")
print(f"    Figures -> {output_dir}")
print("=" * 72)